# H15 · Source check: do Yahoo Finance and ESEF filings agree on the same company?

> **Hypothesis H15:** For the same company and fiscal year, ROA from Yahoo Finance (an unofficial, aggregated feed) and from ESEF (the company's own XBRL filing) agree closely.

| | |
|---|---|
| **Why we ask** | Everything in this project rests on two sources that were never matched to each other. If Yahoo's numbers are noisy or systematically different (for example in some countries), then findings such as H0 may partly reflect data quality, not countries. ESEF filings are official, so they serve as the reference. |
| **Prediction** | **Strong positive** rank correlation (Spearman ρ ≥ 0.8) between the two sources' ROA for matched company-years, and no systematic gap that depends on the country's development. |
| **Data** | Companies matched by country and normalised name (legal suffixes, punctuation and case removed), then a conservative fuzzy match (similarity ≥ 0.93). Matched company-years need ROA in both sources for the same fiscal year. |
| **Primary test (fixed before looking at the result)** | Spearman ρ of ROA (pp) between the two sources over all matched company-years; countries are clustered for the robustness share. |
| **Verdict rule** | *Supported* if the effect has the predicted sign, the wild-bootstrap p < 0.05, and the sign holds in at least 80% of the robustness samples. *Partly supported* if it has the predicted sign and p < 0.10 (or p < 0.05 but robustness is below 80%). *Contradicted* if significantly the opposite way. Otherwise *not supported*. For this hypothesis *supported* additionally requires ρ ≥ 0.8; with 0.5 ≤ ρ < 0.8 the verdict is at most *partly supported*. *Robust share* is the share of countries (with ≥ 8 matched pairs) where ρ ≥ 0.7. |

**Honest framing.** Several hypotheses in this series (H1 to H3) were suggested by an exploratory scan of the same data, so their p-values are optimistic and a pass is a lead, not a confirmation. Where possible the result is **replicated on ESEF**, an independent source (which still shares some companies with Yahoo). All hypotheses are corrected for multiple testing together in `1.0_hypotheses.ipynb`. Everything is **association, not causation**.

In [1]:
import sys, pathlib, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import statsmodels.formula.api as smf
import patsy

warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src" / "analysis"))
import prep, utils
from utils import CONTROLS, fit_cluster, wild_cluster, spearman, boot_ci, partial_rho, bh, scan

con = prep.connect()
q = lambda sql: con.sql(sql).df()
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 120, "display.float_format", "{:,.3f}".format)
px.defaults.template = "plotly_white"

country = prep.load_country(con)
dim, profile, profile_all, panel = country["dim"], country["profile"], country["profile_all"], country["panel"]
feat_cols = country["feat_cols"]
fy = prep.load_firms(con, country)               # Yahoo company-years (primary sample)
names = dim.name
MIN_FIRMS = prep.MIN_FIRMS
print(f"{len(dim)} countries | {len(feat_cols)} country features | {len(fy):,} Yahoo company-years of {fy.company_id.nunique():,} companies")

34 countries | 35 country features | 10,220 Yahoo company-years of 2,590 companies


## 1 · Matching the companies
Names are lower-cased, legal-form words (SA, AG, N.V., S.p.A., PLC …) and punctuation removed. Within each country: exact match on the normalised name first, then fuzzy matches (Python `difflib` ratio ≥ 0.93) for what is left, each ESEF company used once.

In [2]:
import re, difflib
STOP = set("se sa ag plc nv n v ab asa oyj spa s p a ltd limited group holding holdings corp corporation inc co company de the publ kgaa kg as sab cv gmbh sas bv oy aps".split())
def norm(x):
    x = re.sub(r"&", " and ", str(x).lower()); x = re.sub(r"[^a-z0-9 ]", " ", x)
    return " ".join(w for w in x.split() if w not in STOP)
d = q("SELECT company_id, source_id, name, iso3 FROM dim_company"); d["n"] = d.name.map(norm)
yf, es = d[d.source_id == "YF"], d[d.source_id == "ESEF"]
exact = yf.merge(es, on=["iso3", "n"], suffixes=("_y", "_e"))[["company_id_y", "company_id_e", "iso3", "name_y", "name_e"]].assign(tier="exact")
exact = exact.drop_duplicates("company_id_y").drop_duplicates("company_id_e")
left_y = yf[~yf.company_id.isin(exact.company_id_y)]; left_e = es[~es.company_id.isin(exact.company_id_e)]
rows = []
for iso, ey in left_y.groupby("iso3"):
    ee = left_e[left_e.iso3 == iso]
    for _, r in ey.iterrows():
        best = max(((difflib.SequenceMatcher(None, r.n, x.n).ratio(), x) for _, x in ee.iterrows() if x.company_id not in {a["company_id_e"] for a in rows}), default=(0, None), key=lambda t: t[0])
        if best[0] >= 0.93 and len(r.n) > 4:
            rows.append({"company_id_y": r.company_id, "company_id_e": best[1].company_id, "iso3": iso, "name_y": r["name"], "name_e": best[1]["name"], "tier": "fuzzy"})
match = pd.concat([exact, pd.DataFrame(rows)], ignore_index=True)
print(f"{len(match)} matched companies ({(match.tier == 'exact').sum()} exact, {(match.tier == 'fuzzy').sum()} fuzzy) in {match.iso3.nunique()} countries, out of {len(yf)} Yahoo and {len(es)} ESEF companies.")
print("Fuzzy matches to eyeball:"); print(match[match.tier == "fuzzy"][["name_y", "name_e", "iso3"]].head(12).to_string(index=False))
match.groupby("iso3").size().sort_values(ascending=False).to_frame("matched companies").T

498 matched companies (471 exact, 27 fuzzy) in 19 countries, out of 2590 Yahoo and 1312 ESEF companies.
Fuzzy matches to eyeball:
                                                name_y                                                 name_e iso3
                            Kofola CeskoSlovensko a.s.                             Kofola ČeskoSlovensko a.s.  CZE
                                 Philip Morris CR a.s.                                  Philip Morris ČR a.s.  CZE
Atresmedia Corporación de Medios de Comunicación, S.A. ATRESMEDIA CORPORACION DE MEDIOS DE COMUNICACION, S.A.  ESP
         Corporación Acciona Energías Renovables, S.A.             Corporacion Acciona Energias Renovables SA  ESP
                        Árima Real Estate SOCIMI, S.A.                         ARIMA REAL ESTATE SOCIMI, S.A.  ESP
                                 Clínica Baviera, S.A.                                     CLINICA BAVIERA SA  ESP
     Distribuidora Internacional de Alimentación, S.A.         DI

iso3,BEL,ESP,DNK,NLD,FIN,ITA,AUT,FRA,ROU,SWE,PRT,GRC,LTU,POL,EST,LUX,CZE,HUN,LVA
matched companies,64,63,62,58,58,36,24,24,22,22,15,11,8,8,7,6,6,3,1


## 2 · Matched company-years
Both sources' ROA for the same company and fiscal year. ESEF is the reference.

In [3]:
fe = prep.load_firms(con, country, source="ESEF")
a = fy[["company_id", "fiscal_year", "roa_pp", "net_income", "total_assets", "revenue", "currency", "sector", "size_rank", "dev_z", "inst_z"]].rename(columns=lambda c: c if c in ("fiscal_year",) else c + "_y")
b = fe.groupby(["company_id", "fiscal_year"]).agg(roa_e=("roa_pp", "mean"), net_income_e=("net_income", "mean"), total_assets_e=("total_assets", "mean"), revenue_e=("revenue", "mean")).reset_index()
pairs = (match.merge(a, left_on="company_id_y", right_on="company_id_y").merge(b, left_on=["company_id_e", "fiscal_year"], right_on=["company_id", "fiscal_year"]))
pairs = pairs.dropna(subset=["roa_pp_y", "roa_e"]).copy()
pairs["diff"] = pairs.roa_e - pairs.roa_pp_y
print(f"{len(pairs):,} matched company-years, {pairs.company_id_y.nunique()} companies, {pairs.iso3.nunique()} countries, fiscal years {pairs.fiscal_year.min()}–{pairs.fiscal_year.max()}")
fig = px.scatter(pairs, x="roa_pp_y", y="roa_e", color="iso3", hover_data=["name_y", "fiscal_year"], opacity=.6, title="ROA (pp): Yahoo vs ESEF for the same company-year", labels={"roa_pp_y": "Yahoo ROA (pp)", "roa_e": "ESEF ROA (pp)"}, height=520)
fig.add_shape(type="line", x0=-25, y0=-25, x1=25, y1=25, line=dict(color="black", dash="dot")); fig.update_layout(showlegend=False); fig.show()

1,471 matched company-years, 484 companies, 18 countries, fiscal years 2021–2025


## 3 · Primary test: agreement

In [4]:
# winsorise both at 1/99 for a robust view of agreement in levels; rank correlation is unaffected
rho, p, n = spearman(pairs.roa_pp_y, pairs.roa_e)
lo, hi = boot_ci(pairs.roa_pp_y, pairs.roa_e, n=1000)
ae = pairs["diff"].abs()
print(f"Spearman ρ = {rho:+.3f} (95% CI {lo:+.3f} to {hi:+.3f}), p = {p:.2g}, n = {n:,} company-years.")
print(f"Median difference (ESEF − Yahoo) = {pairs['diff'].median():+.2f} pp; median |difference| = {ae.median():.2f} pp; {np.mean(ae <= 1):.0%} of pairs within 1 pp, {np.mean(ae <= 3):.0%} within 3 pp.")
by_c = pairs.groupby("iso3").apply(lambda g: pd.Series({"pairs": len(g), "ρ": spearman(g.roa_pp_y, g.roa_e, 8)[0], "median diff (pp)": g["diff"].median(), "median |diff| (pp)": g["diff"].abs().median()})).query("pairs >= 8").sort_values("ρ")
fig = px.bar(by_c.reset_index(), x="iso3", y="ρ", title="Agreement by country (Spearman ρ of ROA, countries with at least 8 pairs)"); fig.add_hline(y=0.8, line_dash="dot", annotation_text="0.8"); fig.update_layout(height=360); fig.show()
by_c.round(2)

Spearman ρ = +0.992 (95% CI +0.984 to +0.998), p = 0, n = 1,471 company-years.
Median difference (ESEF − Yahoo) = +0.00 pp; median |difference| = 0.00 pp; 97% of pairs within 1 pp, 98% within 3 pp.


,pairs,ρ,median diff (pp),median |diff| (pp)
iso3,,,,
AUT,85.000,0.930,0.000,0.000
ROU,23.000,0.960,0.000,0.000
ESP,187.000,0.980,0.000,0.000
DNK,144.000,0.980,0.000,0.000
BEL,187.000,1.000,0.000,0.000
GRC,31.000,1.000,0.000,0.000
PRT,25.000,1.000,0.000,0.000
ITA,126.000,1.000,0.000,0.000
SWE,65.000,1.000,0.000,0.000


### Which numbers differ?
ROA is net income divided by total assets. Comparing each ingredient: the ratio of ESEF to Yahoo, median over matched company-years (1.00 = identical).

In [5]:
cmp_ = pd.DataFrame({k: (pairs[f"{k}_e"] / pairs[f"{k}_y"]).replace([np.inf, -np.inf], np.nan).dropna() for k in ["net_income", "total_assets", "revenue"]})
summary = pd.DataFrame({"median ratio ESEF/Yahoo": cmp_.median(), "share within ±1%": (np.abs(cmp_ - 1) < 0.01).mean(), "share within ±10%": (np.abs(cmp_ - 1) < 0.10).mean(), "pairs": cmp_.count()})
summary

,median ratio ESEF/Yahoo,share within ±1%,share within ±10%,pairs
net_income,1.000,0.943,0.969,1470
total_assets,1.000,0.971,0.986,1471
revenue,1.000,0.748,0.789,1214


### Is the disagreement related to country development?
If the gap between sources changes with the development index, the measurement error is not random across countries, and could bias comparisons between countries such as H0.

In [6]:
gap = wild_cluster("dev_z_y", pairs.assign(sector="x"), outcome="diff", controls="C(fiscal_year)", B=1999)
print(f"+1 SD development → {gap['coef']:+.2f} pp change in (ESEF − Yahoo) ROA (95% CI {gap['ci_low']:+.2f} to {gap['ci_high']:+.2f}), wild p = {gap['p_wild']:.3f}, {gap['clusters']} countries.")

+1 SD development → -0.17 pp change in (ESEF − Yahoo) ROA (95% CI -0.42 to +0.09), wild p = 0.245, 18 countries.


## 4 · Does the main finding depend on the source? (same companies)
The H0 test (development index → ROA, size and year controls; sector control only in Yahoo, since ESEF has none) re-estimated **on the matched companies only**, once with each source's ROA.

In [7]:
pp = pairs.copy(); pp["size_pct"] = pp.size_rank_y
rows = []
for lab, out in [("Yahoo ROA, matched companies", "roa_pp_y"), ("ESEF ROA, matched companies", "roa_e")]:
    r = wild_cluster("dev_z_y", pp, outcome=out, controls="size_pct + C(fiscal_year)", B=1999)
    rows.append({"source": lab, "coef (pp per +1 SD)": r["coef"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"], "countries": r["clusters"], "firm-years": r["n"]})
src = pd.DataFrame(rows).set_index("source"); src

,coef (pp per +1 SD),ci_low,ci_high,p (wild),countries,firm-years
source,,,,,,
"Yahoo ROA, matched companies",0.230,-1.460,1.920,0.820,18,1471
"ESEF ROA, matched companies",0.173,-1.742,2.088,0.874,18,1471


## 5 · Robustness
Agreement in subsets: exact matches only, excluding financials, and years separately.

In [8]:
def agree(df): 
    r, p_, n_ = spearman(df.roa_pp_y, df.roa_e); return {"ρ": r, "n": n_, "median |diff| (pp)": df["diff"].abs().median()}
subs = {"all matched": pairs, "exact name matches only": pairs[pairs.tier == "exact"], "fuzzy matches only": pairs[pairs.tier == "fuzzy"],
        "excluding financials & real estate": pairs[~pairs.sector_y.isin(["Financial Services", "Real Estate"])],
        "excluding the 3 most-matched countries": pairs[~pairs.iso3.isin(pairs.iso3.value_counts().index[:3])]}
for y_ in sorted(pairs.fiscal_year.unique()):
    if (pairs.fiscal_year == y_).sum() >= 30: subs[f"fiscal year {y_}"] = pairs[pairs.fiscal_year == y_]
rob = pd.DataFrame({k: agree(v) for k, v in subs.items()}).T
robust_share = float((by_c["ρ"] >= 0.7).mean())
rob

,ρ,n,median |diff| (pp)
all matched,0.992,"1,471.000",0.000
exact name matches only,0.995,"1,398.000",0.000
fuzzy matches only,0.935,73.000,0.000
excluding financials & real estate,0.995,"1,157.000",0.000
excluding the 3 most-matched countries,0.986,852.000,0.000
fiscal year 2022,0.985,375.000,0.000
fiscal year 2023,1.000,401.000,0.000
fiscal year 2024,0.988,420.000,0.000
fiscal year 2025,1.000,267.000,0.000


## 6 · Verdict

In [9]:
v = utils.verdict(rho, p, 1, robust_share, int(by_c.shape[0]))
if v == "supported" and rho < 0.8:
    v = "partly supported"
if rho < 0.5:
    v = "not supported"
src_y, src_e = src.iloc[0], src.iloc[1]
rep_same = bool(np.sign(src_e["coef (pp per +1 SD)"]) == np.sign(src_y["coef (pp per +1 SD)"]))
lines = [
 f"Primary: Spearman ρ = {rho:+.3f} between Yahoo and ESEF ROA over {n:,} matched company-years ({pairs.company_id_y.nunique()} companies, {pairs.iso3.nunique()} countries); median |difference| {ae.median():.2f} pp, {np.mean(ae <= 1):.0%} within 1 pp.",
 f"Countries (≥ 8 pairs): ρ ≥ 0.7 in {robust_share:.0%} of {len(by_c)}; lowest {by_c['ρ'].min():+.2f} ({by_c['ρ'].idxmin()}). Ingredient ratios ESEF/Yahoo (median): net income {summary.loc['net_income', 'median ratio ESEF/Yahoo']:.2f}, assets {summary.loc['total_assets', 'median ratio ESEF/Yahoo']:.2f}, revenue {summary.loc['revenue', 'median ratio ESEF/Yahoo']:.2f}.",
 f"Gap vs development: {gap['coef']:+.2f} pp per SD (wild p = {gap['p_wild']:.3f}). The development–ROA coefficient on the same companies: Yahoo {src_y['coef (pp per +1 SD)']:+.2f} (p = {src_y['p (wild)']:.3f}), ESEF {src_e['coef (pp per +1 SD)']:+.2f} (p = {src_e['p (wild)']:.3f}).",
 f"Implication: agreement this close means Yahoo is, for these companies, effectively reproducing the filed accounts, so ESEF replications are NOT independent for the {pairs.company_id_y.nunique()} overlapping companies ({pairs.company_id_y.nunique() / es.shape[0]:.0%} of ESEF companies). They are independent only for the rest.",
 f"VERDICT H15: {v.upper()}",
]
print("\n".join(lines))
utils.save_result("H15", id="H15", title="Source agreement: Yahoo vs ESEF", statement="Yahoo and ESEF give the same ROA for the same company-year.",
                  expected_sign=1, effect=float(rho), ci_low=float(lo), ci_high=float(hi), effect_unit="Spearman ρ of ROA, Yahoo vs ESEF (matched company-years)",
                  p_primary=float(max(p, 1e-12)), p_method="Spearman test", n=int(n), clusters=int(by_c.shape[0]), robust_share=robust_share,
                  replication_coef=float(src_e["coef (pp per +1 SD)"]), replication_p=float(src_e["p (wild)"]), replication_same_sign=rep_same, verdict=v,
                  notebook="1.15_source_check_yahoo_vs_esef.ipynb", notes=lines)

Primary: Spearman ρ = +0.992 between Yahoo and ESEF ROA over 1,471 matched company-years (484 companies, 18 countries); median |difference| 0.00 pp, 97% within 1 pp.
Countries (≥ 8 pairs): ρ ≥ 0.7 in 100% of 15; lowest +0.93 (AUT). Ingredient ratios ESEF/Yahoo (median): net income 1.00, assets 1.00, revenue 1.00.
Gap vs development: -0.17 pp per SD (wild p = 0.245). The development–ROA coefficient on the same companies: Yahoo +0.23 (p = 0.820), ESEF +0.17 (p = 0.874).
Implication: agreement this close means Yahoo is, for these companies, effectively reproducing the filed accounts, so ESEF replications are NOT independent for the 484 overlapping companies (37% of ESEF companies). They are independent only for the rest.
VERDICT H15: SUPPORTED


### Limits
- Matching is by company **name**, with no common identifier in our data; a few wrong or missed matches are possible (the fuzzy tier is shown for inspection).
- Matched companies are EU-listed companies present in both sources (about 19 countries), so the check does not cover non-EU countries, where Yahoo is the only source.
- ESEF is the company's own filing, but taxonomy tags can be chosen differently by filers; "agree" does not prove either is right.
- Association, not causation: this notebook checks measurement, not the research question.

In [10]:
con.close()